In [ ]:
%pip install -r requirements-offline.txt --quiet

# Solution — Module M5: สถิติที่ใช้ได้จริง

เปิดเผยหลังจบแล็บ — เทียบกับของตัวเอง โฟกัสที่**เหตุผล** ไม่ใช่ตัวเลขตรงกัน

In [ ]:
import pandas as pd
from pathlib import Path


def repo_root() -> Path:
    """หา repo root โดยไล่ขึ้นจาก cwd."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "requirements-offline.txt").exists():
            return p
    raise SystemExit(
        "ไม่พบ repo root — เปิด JupyterLab จาก repo root (uv run jupyter lab)")

DATA = repo_root() / "datasets" / "palmer_penguins"  # noqa: E305

penguins = pd.read_csv(DATA / "penguins.csv").dropna()

## ส่วนที่ 1 — ค่ากลาง: mean vs median

In [ ]:
median_bill = penguins["bill_length_mm"].median()
std_bill = penguins["bill_length_mm"].std()
print(f"median = {median_bill:.1f}, std = {std_bill:.1f}")

**1.1** mean — ขยับตามค่า outlier ทันที (ค่า 15 กก. เข้าไปอยู่ในผลรวม)
ส่วน median ขยับเพียง "ตำแหน่งกลาง" เลื่อนครึ่งแถว จึงแทบไม่ขยับ
นี่คือเหตุผลที่ข้อมูลที่มีค่าสุดขอบต้องรายงานด้วย median

**1.2** ตัวอย่างคำตอบที่ดี: "median + std (หรือช่วงกลาง 50%) + จำนวนแถว"
เพราะ mean ตัวเดียวถูก outlier ลากได้ และตัวเลขไม่มี n ก็พิสูจน์ไม่ได้ว่า
มาจากข้อมูลจริงกี่ตัว

## ส่วนที่ 2 — สรุปต่อกลุ่ม (groupby + สถิติ)

In [ ]:
summary = penguins.groupby("species").agg(
    n=("flipper_length_mm", "size"),
    mean_mm=("flipper_length_mm", "mean"),
    median_mm=("flipper_length_mm", "median"),
    std_mm=("flipper_length_mm", "std"),
)
summary.round(1)

**2.1** Gentoo แผ่นปีกยาวสุด (~217 มม.) · กระจายมากสุดคือ Gentoo เช่นกัน
(std ~6.6 มม.) — แปลเป็นภาษาคน: "ปีก Gentoo ยาวกว่าเพื่อนชัดเจน
และตัวต่อตัวต่างกันพอสมควร ไม่ได้ยาวเท่ากันหมด

## ส่วนที่ 3 — รูปทรงข้อมูล (histogram)

In [ ]:
import matplotlib.pyplot as plt  # noqa: E402

plt.rcParams["font.family"] = [
    "DejaVu Sans", "Noto Sans Thai", "Leelawadee UI", "Tahoma",
]

fig, ax = plt.subplots(figsize=(7, 3))
penguins["bill_depth_mm"].plot.hist(bins=30, ax=ax, color="#457b9d")
ax.set_title("ความลึกจะงอยปาก (มม.)")
ax.set_xlabel("bill_depth_mm")
plt.show()

**3.1** เป็นสองยอดชัดเจน — ตัวแบ่งคือ**ชนิดพันธุ์** (ตรวจได้ด้วย
`penguins.groupby("species")["bill_depth_mm"].median()`:
Adelie/Chinstrap ~18 มม. แต่ Gentoo ~15 มม.)
จึงได้บทเรียนเดียวกับน้ำหนัก: mean รวมตกอยู่ร่องกลาง ไม่ตรงกับตัวจริง

## ส่วนที่ 4 — โบนัสไทย: ราคาข้าวหอมมะลิ

In [ ]:
rice = pd.read_csv(
    repo_root() / "datasets" / "rice_price_105" / "rice105_monthly.csv")

by_year = rice.groupby("year").agg(
    median_price=("Value", "median"),
    max_price=("Value", "max"),
)
by_year.round(0)

**4.1** พ.ศ. 2567 (2024) — max สูงสุดที่ 15,686 บาท/ตัน สำหรับตัวเลขเดียว
แทนทั้งปี ควรใช้ **median** เพราะทนต่อเดือนที่ราคาพุ่งผิดปกติกว่า mean
ข้อพึงระวัง: ปี 2568 มีเพียง ม.ค.–ส.ค. (8 เดือน) จึงเทียบกับปีเต็มไม่ได้ —
การเช็คว่าแต่ละกลุ่มมีข้อมูลกี่แถว (`n`) มาจากบทเรียน M5 ข้อ 4 เสมอ